# MRI → PET Preprocessing Pipeline


1. ZIP Extraction & DICOM → NIfTI Conversion
2. N4 Bias Field Correction (removes scanner intensity inhomogeneity)
3. Skull Stripping (removes non-brain tissue)
4. MNI152 Template Registration (aligns all brains to standard space)
5. PET → MRI Registration (aligns PET to MRI)
6. Brain Masking (zeros out non-brain regions)
7. PET Intensity Scaling (SUVR-like normalization)
8. Intensity Normalization (percentile clipping + scaling to [-1, 1])
9. Resampling to fixed shape using SimpleITK
10. Optional PET Smoothing

## Install Dependencies



In [41]:
# Install core dependencies
!pip install SimpleITK nibabel nilearn scipy numpy

# Install HD-BET for skull stripping (optional but recommended)
# Requires PyTorch to be installed first
!pip install hd-bet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports and Configuration

In [3]:
import os
import gc
import subprocess
import zipfile
import tempfile
import logging
import sys
from datetime import datetime

import numpy as np
import nibabel as nib
import SimpleITK as sitk

# Check if HD-BET is available for skull stripping
try:
    from HD_BET.run import run_hd_bet
    HAS_HD_BET = True
    print("HD-BET is available — will use for skull stripping.")
except ImportError:
    HAS_HD_BET = False
    print("HD-BET not found — will use Otsu threshold fallback for skull stripping.")

# Check if nilearn is available for MNI template
try:
    from nilearn.datasets import load_mni152_template
    HAS_NILEARN = True
    print("nilearn is available — will auto-download MNI152 template.")
except ImportError:
    HAS_NILEARN = False
    print("nilearn not found — you must provide MNI template path manually.")

HD-BET is available — will use for skull stripping.
nilearn is available — will auto-download MNI152 template.


# CONFIGURATION

In [ ]:
# Input/output directories
INPUT_BASE = r"D:\Organized_New_Only\Normal"
OUTPUT_BASE = r"D:\Organized_New_Only\Normal\Normal_Preprocessed_Enhanced"
LOG_FILE = "enhanced_preprocessing_log.txt"

# Number of patients to process
NUM_PATIENTS = 200

# Target output shape (depth, height, width)
TARGET_SHAPE = (160, 192, 160)

# Target voxel spacing in mm (isotropic 1mm is standard)
TARGET_SPACING = (1.0, 1.0, 1.0)

# Path to MNI152 template (leave as None to auto-download with nilearn)
MNI_TEMPLATE_PATH = None

# Skull stripping method: 'hd-bet' or 'otsu'
# 'hd-bet' is much better but requires the HD-BET package
# SKULL_STRIP_METHOD = 'hd-bet' if HAS_HD_BET else 'otsu'
SKULL_STRIP_METHOD = 'otsu'

# Normalization method: 'percentile' or 'zscore'
NORMALIZATION_METHOD = 'percentile'

# PET smoothing FWHM in mm (set to 0 to disable)
PET_SMOOTHING_FWHM = 4.0

# PET tracer selection priority
PET_TRACER_PRIORITY = [
    "AMYLOID", "FLORBETABEN", "PIB", "FDG",
    "TAU", "FLORTAUCIPIR", "AV1451", "FLORBETAPIR"
]

# Temporary directory for intermediate files (skull stripping etc.)
TEMP_DIR = os.path.join(OUTPUT_BASE, "_temp")
os.makedirs(TEMP_DIR, exist_ok=True)
os.makedirs(OUTPUT_BASE, exist_ok=True)

print(f"Input:  {INPUT_BASE}")
print(f"Output: {OUTPUT_BASE}")
print(f"Target shape:   {TARGET_SHAPE}")
print(f"Target spacing: {TARGET_SPACING}")
print(f"Skull stripping: {SKULL_STRIP_METHOD}")
print(f"Normalization:   {NORMALIZATION_METHOD}")
print(f"PET smoothing:   {PET_SMOOTHING_FWHM} mm FWHM")

Input:  D:\Organized_New_Only\Normal
Output: D:\Organized_New_Only\Normal\Normal_Preprocessed_Enhanced
Target shape:   (160, 192, 160)
Target spacing: (1.0, 1.0, 1.0)
Skull stripping: otsu
Normalization:   percentile
PET smoothing:   4.0 mm FWHM


## 2. Logging Setup

In [5]:
# Setup logging to both file and console
logger = logging.getLogger("preprocessing")
logger.setLevel(logging.INFO)

# Clear any existing handlers
if logger.handlers:
    logger.handlers.clear()

# File handler
file_handler = logging.FileHandler(LOG_FILE, encoding='utf-8')
file_handler.setLevel(logging.INFO)

# Console handler
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)

# Formatter
formatter = logging.Formatter('[%(asctime)s] %(levelname)s: %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.addHandler(console_handler)


def log(message):
    """Shortcut for logging."""
    logger.info(message)


log("Logging initialized.")

[2026-03-16 15:52:49] INFO: Logging initialized.


## 3. Utility Functions — File Finding and Scan Selection

These functions find NIfTI files and select the best MRI/PET scans for each patient.

In [6]:
def is_resource_fork(filename):
    """Check if a file is a macOS resource fork (should be skipped)."""
    basename = os.path.basename(filename)
    return basename.startswith('._') or basename.startswith('.')


def find_all_nifti(directory):
    """
    Find all valid NIfTI files in a directory tree.
    Returns file paths sorted by size (largest first),
    because larger files are more likely to be full 3D volumes.
    """
    if not os.path.exists(directory):
        return []

    candidates = []
    for root, dirs, files in os.walk(directory):
        # Skip hidden/resource fork directories
        dirs[:] = [d for d in dirs if not is_resource_fork(d)]
        for file in files:
            if is_resource_fork(file):
                continue
            if file.endswith('.nii.gz') or file.endswith('.nii'):
                full_path = os.path.join(root, file)
                try:
                    # Quick validation — check if header loads
                    img = nib.load(full_path)
                    _ = img.header
                    size = os.path.getsize(full_path)
                    candidates.append((full_path, size))
                except Exception:
                    continue

    # Sort by file size (largest first)
    candidates.sort(key=lambda x: x[1], reverse=True)
    return [c[0] for c in candidates]


def choose_mri(mri_dir):
    """
    Choose the best MRI scan from a directory.
    Priority: MPRAGE > T1/SPGR > non-FLAIR > largest file.
    T1-weighted MRI gives the best brain anatomy for registration.
    """
    all_nii = find_all_nifti(mri_dir)
    if not all_nii:
        return None

    # Priority 1: MPRAGE (best T1 sequence)
    for f in all_nii:
        upper = (os.path.basename(f) + os.path.dirname(f)).upper()
        if "MPRAGE" in upper:
            log(f"    MRI selected (MPRAGE): {os.path.basename(f)}")
            return f

    # Priority 2: T1-weighted (not FLAIR)
    for f in all_nii:
        upper = (os.path.basename(f) + os.path.dirname(f)).upper()
        if ("T1" in upper or "SPGR" in upper or "IR-FSPGR" in upper) and "FLAIR" not in upper:
            log(f"    MRI selected (T1): {os.path.basename(f)}")
            return f

    # Priority 3: Anything that's not FLAIR/T2
    for f in all_nii:
        upper = (os.path.basename(f) + os.path.dirname(f)).upper()
        if "FLAIR" not in upper and "T2" not in upper:
            log(f"    MRI selected (non-FLAIR): {os.path.basename(f)}")
            return f

    # Fallback: largest file
    log(f"    MRI selected (fallback/largest): {os.path.basename(all_nii[0])}")
    return all_nii[0]


def choose_pet(pet_dir):
    """
    Choose the best PET scan from a directory.
    Priority by tracer type, then largest file.
    """
    all_nii = find_all_nifti(pet_dir)
    if not all_nii:
        return None

    # Try each tracer in priority order
    for tracer in PET_TRACER_PRIORITY:
        for f in all_nii:
            upper = (os.path.basename(f) + os.path.dirname(f)).upper()
            if tracer in upper:
                log(f"    PET selected ({tracer}): {os.path.basename(f)}")
                return f

    # Fallback: largest file
    log(f"    PET selected (largest): {os.path.basename(all_nii[0])}")
    return all_nii[0]

## 4. NIfTI Loading with RAS → LPS Conversion

NIfTI files use RAS (Right-Anterior-Superior) orientation, but SimpleITK uses LPS (Left-Posterior-Superior). This function handles the conversion so that spatial metadata (origin, direction, spacing) is preserved correctly.

In [7]:
def nifti_to_sitk(nifti_path):
    """
    Load a NIfTI file and convert it to a SimpleITK image
    with correct spatial metadata (RAS → LPS conversion).
    
    Also handles 4D scans by averaging or taking the first volume.
    """
    nib_img = nib.load(nifti_path)
    data = nib_img.get_fdata().astype(np.float32)

    # Handle 4D scans (e.g., multi-frame PET)
    if data.ndim == 4:
        if data.shape[3] <= 10:
            # Few frames — average them
            data = np.mean(data, axis=-1).astype(np.float32)
        else:
            # Many frames — take the first
            data = data[:, :, :, 0].astype(np.float32)

    affine = nib_img.affine
    spacing = np.abs(nib_img.header.get_zooms()[:3]).tolist()

    # NIfTI stores data as (x,y,z) in RAS
    # SimpleITK expects array in (z,y,x) order
    data_sitk_order = np.transpose(data, (2, 1, 0))
    sitk_img = sitk.GetImageFromArray(data_sitk_order)
    sitk_img.SetSpacing([float(s) for s in spacing])

    # Convert origin from RAS to LPS (flip X and Y axes)
    origin = affine[:3, 3].copy().astype(float)
    origin[0] = -origin[0]
    origin[1] = -origin[1]
    sitk_img.SetOrigin(origin.tolist())

    # Convert direction matrix from RAS to LPS
    rot = affine[:3, :3] / np.array(spacing)
    direction = rot.copy()
    direction[0, :] = -direction[0, :]
    direction[1, :] = -direction[1, :]
    sitk_img.SetDirection(direction.flatten().tolist())

    return sitk_img


def sitk_to_numpy(sitk_image):
    """
    Convert SimpleITK image to numpy array.
    SimpleITK stores as (z,y,x), this returns (x,y,z).
    """
    array = sitk.GetArrayFromImage(sitk_image)
    return np.transpose(array, (2, 1, 0)).astype(np.float32)

## 5. N4 Bias Field Correction

MRI scanners produce **intensity inhomogeneity** — gradual brightness changes across the brain that are not biological. For example, the left side of the brain may appear brighter than the right.

N4 bias field correction removes this scanner artifact, making intensities more uniform across the brain. This is a **standard step** in neuroimaging pipelines.

In [8]:
def n4_bias_correction(sitk_image):
    """
    Apply N4 bias field correction to an MRI image.
    Removes scanner-related intensity inhomogeneity.
    
    Input:  SimpleITK image (MRI)
    Output: Corrected SimpleITK image
    """
    image = sitk.Cast(sitk_image, sitk.sitkFloat32)

    # Create a rough brain mask using Otsu thresholding
    # This tells N4 which voxels to use for estimating the bias field
    mask = sitk.OtsuThreshold(image, 0, 1, 200)

    # Configure and run N4 correction
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    corrector.SetMaximumNumberOfIterations([50, 50, 30, 20])

    corrected = corrector.Execute(image, mask)

    return corrected

## 6. Skull Stripping

Skull stripping removes non-brain tissue so the model can focus on the brain.

In [9]:
def skull_strip_hd_bet(sitk_image, temp_dir):
    """
    Skull stripping using HD-BET (deep learning method).
    Saves to temp file, runs HD-BET, loads result.
    
    Returns: (brain_image, brain_mask) as SimpleITK images
    """
    # HD-BET works on NIfTI files, so save a temp file
    temp_input = os.path.join(temp_dir, "temp_mri_input.nii.gz")
    temp_output = os.path.join(temp_dir, "temp_mri_brain.nii.gz")
    sitk.WriteImage(sitk_image, temp_input)

    # Run HD-BET (mode='fast' for speed, device='cpu' if no GPU)
    run_hd_bet(temp_input, temp_output, mode='fast', device='cpu')

    # Load the brain-extracted image and mask
    brain = sitk.ReadImage(temp_output, sitk.sitkFloat32)
    mask_path = temp_output.replace(".nii.gz", "_mask.nii.gz")
    mask = sitk.ReadImage(mask_path, sitk.sitkFloat32)

    # Clean up temp files
    for f in [temp_input, temp_output, mask_path]:
        if os.path.exists(f):
            os.remove(f)

    return brain, mask


def skull_strip_otsu(sitk_image):
    """
    Simple skull stripping fallback using Otsu thresholding.
    Less accurate than HD-BET but requires no extra packages.
    
    Steps:
    1. Otsu threshold to separate brain from background
    2. Morphological closing to fill holes
    3. Keep only the largest connected component (the brain)
    4. Apply mask to the image
    
    Returns: (brain_image, brain_mask) as SimpleITK images
    """
    image = sitk.Cast(sitk_image, sitk.sitkFloat32)

    # Step 1: Otsu threshold
    mask = sitk.OtsuThreshold(image, 0, 1)

    # Step 2: Morphological closing to fill small holes in the mask
    mask = sitk.BinaryMorphologicalClosing(mask, [5, 5, 5])

    # Step 3: Keep only the largest connected component
    labeled = sitk.ConnectedComponent(mask)
    stats = sitk.LabelShapeStatisticsImageFilter()
    stats.Execute(labeled)

    labels = stats.GetLabels()
    if labels:
        largest_label = max(labels, key=lambda l: stats.GetNumberOfPixels(l))
        mask = sitk.Equal(labeled, largest_label)

    mask = sitk.Cast(mask, sitk.sitkFloat32)

    # Step 4: Apply mask to the image
    brain = sitk.Multiply(image, mask)

    return brain, mask


def skull_strip(sitk_image, method='hd-bet', temp_dir=None):
    """
    Skull strip an MRI image.
    Dispatches to HD-BET or Otsu fallback based on method.
    """
    if method == 'hd-bet' and HAS_HD_BET:
        return skull_strip_hd_bet(sitk_image, temp_dir)
    else:
        return skull_strip_otsu(sitk_image)

## 7. MNI152 Template Registration

Different patients have different **brain sizes, orientations, and positions**. Without spatial normalization, the model must learn to handle all this variation.

Registering every MRI to the **MNI152 standard template** puts all brains into the **same coordinate space**, greatly improving learning.

The pipeline becomes:
```
MRI → MNI space
PET → MRI space → MNI space
```

In [10]:
def load_mni_template(mni_path=None):
    """
    Load the MNI152 1mm template as a SimpleITK image.
    
    If mni_path is provided, loads from that file.
    Otherwise, downloads it using nilearn.
    """
    if mni_path and os.path.exists(mni_path):
        log(f"Loading MNI template from: {mni_path}")
        return sitk.ReadImage(mni_path, sitk.sitkFloat32)

    if not HAS_NILEARN:
        raise RuntimeError(
            "nilearn not installed and no MNI template path provided. "
            "Install nilearn (pip install nilearn) or provide MNI_TEMPLATE_PATH."
        )

    log("Downloading MNI152 template via nilearn...")
    mni_nib = load_mni152_template(resolution=1)

    # Save to temp file and reload with SimpleITK
    # (nilearn returns nibabel format, we need SimpleITK)
    temp_path = os.path.join(TEMP_DIR, "mni152_1mm.nii.gz")
    nib.save(mni_nib, temp_path)
    mni_sitk = sitk.ReadImage(temp_path, sitk.sitkFloat32)

    log(f"MNI template loaded. Shape: {mni_sitk.GetSize()}, Spacing: {mni_sitk.GetSpacing()}")
    return mni_sitk


# Load the MNI template once (reused for every patient)
MNI_TEMPLATE = load_mni_template(MNI_TEMPLATE_PATH)

[2026-03-16 15:52:49] INFO: Downloading MNI152 template via nilearn...
[2026-03-16 15:52:49] INFO: MNI template loaded. Shape: (197, 233, 189), Spacing: (1.0, 1.0, 1.0)


In [11]:
def register_to_mni(source_sitk, mni_template):
    """
    Register a brain MRI to the MNI152 template using affine registration
    with mutual information.
    
    Input:  source_sitk   — skull-stripped MRI (SimpleITK image)
            mni_template  — MNI152 template (SimpleITK image)
    Output: (registered_image, transform)
            The transform can be reused to bring PET/mask into MNI space.
    """
    fixed = sitk.Cast(mni_template, sitk.sitkFloat32)
    moving = sitk.Cast(source_sitk, sitk.sitkFloat32)

    # Initialize with centre-of-geometry alignment
    initial_tx = sitk.CenteredTransformInitializer(
        fixed, moving,
        sitk.AffineTransform(3),
        sitk.CenteredTransformInitializerFilter.GEOMETRY
    )

    # Setup registration
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=64)
    reg.SetMetricSamplingStrategy(reg.RANDOM)
    reg.SetMetricSamplingPercentage(0.25)
    reg.SetOptimizerAsGradientDescent(
        learningRate=1.0,
        numberOfIterations=200,
        convergenceMinimumValue=1e-7,
        convergenceWindowSize=15
    )
    reg.SetOptimizerScalesFromPhysicalShift()
    reg.SetInterpolator(sitk.sitkLinear)

    # Multi-resolution: coarse to fine
    reg.SetShrinkFactorsPerLevel([8, 4, 2, 1])
    reg.SetSmoothingSigmasPerLevel([4.0, 2.0, 1.0, 0.0])
    reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    reg.SetInitialTransform(initial_tx, inPlace=False)

    try:
        transform = reg.Execute(fixed, moving)
        metric = reg.GetMetricValue()
        log(f"    MNI registration converged: metric = {metric:.6f}")
    except Exception as e:
        log(f"    MNI registration failed: {e}, using centre alignment")
        transform = initial_tx

    # Resample the MRI into MNI space
    registered = resample_with_transform(moving, fixed, transform)

    return registered, transform


def resample_with_transform(image, reference, transform, is_mask=False):
    """
    Resample an image into the reference image's space using a given transform.
    For binary masks, use nearest-neighbor interpolation to keep values 0/1.
    """
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(reference)
    resampler.SetTransform(transform)
    resampler.SetDefaultPixelValue(0)

    if is_mask:
        resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    else:
        resampler.SetInterpolator(sitk.sitkLinear)

    return resampler.Execute(image)

## 8. PET → MRI Registration

PET and MRI are different modalities, so they are not spatially aligned by default. We register PET to the MRI using **mutual information** (appropriate for cross-modality registration).

In [12]:
def register_pet_to_mri(pet_sitk, mri_sitk):
    """
    Register PET to MRI using multi-resolution mutual information.
    
    Input:  pet_sitk — PET image (SimpleITK)
            mri_sitk — MRI image in native space (SimpleITK)
    Output: (registered_pet, transform)
    """
    fixed = sitk.Cast(mri_sitk, sitk.sitkFloat32)
    moving = sitk.Cast(pet_sitk, sitk.sitkFloat32)

    # Initialize with centre-of-geometry alignment
    initial_tx = sitk.CenteredTransformInitializer(
        fixed, moving,
        sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.GEOMETRY
    )

    # Setup registration
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=64)
    reg.SetMetricSamplingStrategy(reg.RANDOM)
    reg.SetMetricSamplingPercentage(0.25)
    reg.SetOptimizerAsGradientDescent(
        learningRate=1.0,
        numberOfIterations=300,
        convergenceMinimumValue=1e-7,
        convergenceWindowSize=15
    )
    reg.SetOptimizerScalesFromPhysicalShift()
    reg.SetInterpolator(sitk.sitkLinear)

    # Multi-resolution: coarse to fine
    reg.SetShrinkFactorsPerLevel([8, 4, 2, 1])
    reg.SetSmoothingSigmasPerLevel([4.0, 2.0, 1.0, 0.0])
    reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    reg.SetInitialTransform(initial_tx, inPlace=False)

    try:
        transform = reg.Execute(fixed, moving)
        metric = reg.GetMetricValue()
        log(f"    PET→MRI registration converged: metric = {metric:.6f}")
    except Exception as e:
        log(f"    PET→MRI registration failed: {e}, using centre alignment")
        transform = initial_tx

    # Resample PET into MRI space
    registered = resample_with_transform(moving, fixed, transform)

    return registered, transform

## 9. Brain Masking

- Removes any residual background noise
- Ensures the model only sees brain tissue
- Zeros out non-brain regions

In [13]:
def apply_brain_mask(sitk_image, sitk_mask):
    """
    Apply a binary brain mask to an image.
    Non-brain voxels are set to zero.
    
    Input:  sitk_image — the image to mask (SimpleITK)
            sitk_mask  — binary brain mask (SimpleITK, values 0 or 1)
    Output: masked SimpleITK image
    """
    image = sitk.Cast(sitk_image, sitk.sitkFloat32)
    mask = sitk.Cast(sitk_mask, sitk.sitkFloat32)

    # Element-wise multiplication: brain * 1 = brain, background * 0 = 0
    masked = sitk.Multiply(image, mask)

    return masked

## 10. Intensity Normalization

### MRI/PET Normalization (two options)
- **Percentile normalization**: Clips outliers using percentiles, scales to [-1, 1]. Good for diffusion models.
- **Z-score normalization**: Subtracts mean and divides by std (computed over brain voxels only). Common in multi-center studies.

In [14]:
def normalize_pet_suvr(pet_array, mask_array):
    """
    Simplified SUVR normalization for PET.
    Divides all PET values by the mean uptake within the brain mask.
    This stabilizes PET intensities across different patients/scanners.
    
    Input:  pet_array  — PET as numpy array
            mask_array — brain mask as numpy array (0s and 1s)
    Output: SUVR-normalized PET numpy array
    """
    brain_voxels = pet_array[mask_array > 0]

    if len(brain_voxels) < 100:
        log("    WARNING: Too few brain voxels for SUVR, skipping.")
        return pet_array

    # Mean brain uptake as the reference value
    reference = np.mean(brain_voxels)

    if reference > 0:
        pet_array = pet_array / reference

    return pet_array


def normalize_percentile(data, low_pct=1.0, high_pct=99.0):
    """
    Percentile normalization to [-1, 1].
    
    Steps:
    1. Zero out negative values
    2. Compute percentile range from non-zero voxels
    3. Clip to percentile range
    4. Scale to [-1, 1]
    """
    data = data.copy()
    data[data < 0] = 0

    nonzero = data[data > 0]
    if len(nonzero) < 100:
        return data

    v_min = np.percentile(nonzero, low_pct)
    v_max = np.percentile(nonzero, high_pct)

    # Clip outliers
    np.clip(data, v_min, v_max, out=data)

    # Scale to [0, 1] then to [-1, 1]
    data = (data - v_min) / (v_max - v_min + 1e-8)
    data = 2.0 * data - 1.0

    return data


def normalize_zscore(data, mask=None):
    """
    Z-score normalization: (data - mean) / std.
    Computed over brain voxels only (from mask) if provided.
    """
    data = data.copy()

    if mask is not None:
        brain_voxels = data[mask > 0]
    else:
        brain_voxels = data[data > 0]

    if len(brain_voxels) < 100:
        return data

    mean = np.mean(brain_voxels)
    std = np.std(brain_voxels)

    if std > 0:
        data = (data - mean) / std

    return data

## 11. Resampling with SimpleITK

 **SimpleITK resampling** resamples to a fixed voxel spacing first, then crops or pads to the target shape. This preserves anatomical proportions correctly.

In [15]:
def resample_to_target(sitk_image, target_shape=TARGET_SHAPE, target_spacing=TARGET_SPACING):
    """
    Resample a SimpleITK image to a fixed voxel spacing, then
    center-crop or center-pad to the target shape.
    
    This is better than scipy.ndimage.zoom because it respects
    voxel spacing and avoids anatomical distortion.
    
    Input:  SimpleITK image
    Output: numpy array with shape = target_shape (x, y, z)
    """
    image = sitk.Cast(sitk_image, sitk.sitkFloat32)

    # --- Step 1: Resample to target voxel spacing ---
    original_spacing = image.GetSpacing()
    original_size = image.GetSize()

    # Calculate new size to keep the same physical extent
    new_size = [
        int(round(osz * osp / tsp))
        for osz, osp, tsp in zip(original_size, original_spacing, target_spacing)
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputDirection(image.GetDirection())
    resampler.SetOutputOrigin(image.GetOrigin())
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetDefaultPixelValue(0)
    resampler.SetTransform(sitk.Transform())  # Identity transform

    resampled = resampler.Execute(image)

    # --- Step 2: Convert to numpy and center-crop/pad ---
    # SimpleITK array is (z, y, x), convert to (x, y, z)
    array = sitk.GetArrayFromImage(resampled)
    array = np.transpose(array, (2, 1, 0)).astype(np.float32)

    result = center_crop_or_pad(array, target_shape)

    return result


def center_crop_or_pad(array, target_shape):
    """
    Center-crop or zero-pad a 3D array to match target_shape.
    
    If the array is larger than target → crop from the center.
    If the array is smaller than target → pad with zeros around the center.
    """
    result = np.zeros(target_shape, dtype=np.float32)

    # Calculate start/end indices for both source and target
    for_source_start = []
    for_source_end = []
    for_target_start = []
    for_target_end = []

    for s, t in zip(array.shape, target_shape):
        if s >= t:
            # Source is larger: crop from center
            start = (s - t) // 2
            for_source_start.append(start)
            for_source_end.append(start + t)
            for_target_start.append(0)
            for_target_end.append(t)
        else:
            # Source is smaller: pad with zeros
            start = (t - s) // 2
            for_source_start.append(0)
            for_source_end.append(s)
            for_target_start.append(start)
            for_target_end.append(start + s)

    # Copy data
    result[
        for_target_start[0]:for_target_end[0],
        for_target_start[1]:for_target_end[1],
        for_target_start[2]:for_target_end[2]
    ] = array[
        for_source_start[0]:for_source_end[0],
        for_source_start[1]:for_source_end[1],
        for_source_start[2]:for_source_end[2]
    ]

    return result

## 12. PET Smoothing 

PET scanners produce **noisy images**. Applying a Gaussian smooth (typically 4–6mm FWHM) reduces noise and matches the effective PET scanner resolution.

In [16]:
def smooth_pet(sitk_image, fwhm_mm=4.0):
    """
    Apply Gaussian smoothing to a PET image.
    
    FWHM (Full Width at Half Maximum) controls the amount of smoothing.
    Typical values: 4-6 mm for PET.
    
    FWHM is converted to sigma: sigma = FWHM / (2 * sqrt(2 * ln(2))) ≈ FWHM / 2.355
    """
    if fwhm_mm <= 0:
        return sitk_image

    # Convert FWHM to sigma (in mm)
    sigma_mm = fwhm_mm / 2.355

    # Apply Gaussian smoothing
    smoothed = sitk.SmoothingRecursiveGaussian(sitk_image, sigma_mm)

    return smoothed

## 13. ZIP Extraction and DICOM → NIfTI Conversion

In [17]:
def extract_zips(patient_ids, input_base):
    """
    Extract all ZIP files for the given patients.
    Skips already-extracted archives.
    """
    extracted_count = 0
    skipped_forks = 0

    for idx, patient_id in enumerate(patient_ids, 1):
        patient_path = os.path.join(input_base, patient_id)

        for modality in ["MRI", "PET"]:
            mod_path = os.path.join(patient_path, modality)
            if not os.path.exists(mod_path):
                continue

            for f in os.listdir(mod_path):
                if not f.endswith('.zip'):
                    continue
                if is_resource_fork(f):
                    skipped_forks += 1
                    continue

                zip_path = os.path.join(mod_path, f)
                extract_dir = os.path.join(mod_path, f.replace('.zip', ''))

                if not os.path.exists(extract_dir):
                    try:
                        os.makedirs(extract_dir, exist_ok=True)
                        with zipfile.ZipFile(zip_path, 'r') as zr:
                            zr.extractall(extract_dir)
                        extracted_count += 1
                    except Exception as e:
                        log(f"ERROR extracting {patient_id}/{f}: {e}")

        if idx % 10 == 0:
            log(f"[{idx}/{len(patient_ids)}] Extraction: {extracted_count} extracted")

    log(f"Extraction complete: {extracted_count} extracted, {skipped_forks} resource forks skipped")
    return extracted_count


def convert_dicom_to_nifti(patient_ids, input_base):
    """
    Convert DICOM folders to NIfTI using dcm2niix.
    Skips folders that already contain NIfTI files.
    """
    # Check if dcm2niix is available
    dcm2niix_cmd = "dcm2niix"
    try:
        r = subprocess.run([dcm2niix_cmd, '--version'], capture_output=True, text=True)
        has_dcm2niix = (r.returncode in (0, 1, 2, 3))
        if has_dcm2niix:
            log(f"Using dcm2niix: {dcm2niix_cmd}")
    except Exception:
        has_dcm2niix = False
        log("dcm2niix not found — skipping DICOM conversion")
        return 0

    nifti_count = 0

    for idx, patient_id in enumerate(patient_ids, 1):
        patient_path = os.path.join(input_base, patient_id)

        for modality in ["MRI", "PET"]:
            mod_path = os.path.join(patient_path, modality)
            if not os.path.exists(mod_path):
                continue

            extracted_dirs = [
                d for d in os.listdir(mod_path)
                if os.path.isdir(os.path.join(mod_path, d))
                and not is_resource_fork(d)
            ]

            for ed in extracted_dirs:
                ep = os.path.join(mod_path, ed)

                # Check if NIfTI already exists
                existing = [
                    f for f in os.listdir(ep)
                    if (f.endswith('.nii.gz') or f.endswith('.nii'))
                    and not is_resource_fork(f)
                ]

                if existing:
                    nifti_count += 1
                    continue

                # Convert DICOM to NIfTI
                try:
                    subprocess.run(
                        [dcm2niix_cmd, '-z', 'y', '-f', ed, '-o', ep, ep],
                        capture_output=True, timeout=120
                    )
                    nifti_count += 1
                except Exception as e:
                    log(f"ERROR converting {patient_id}/{modality}/{ed}: {e}")

        if idx % 10 == 0:
            log(f"[{idx}/{len(patient_ids)}] Conversion: {nifti_count} NIfTI files")

    log(f"Conversion complete: {nifti_count} NIfTI files")
    return nifti_count

## 14. Complete Preprocessing Pipeline

This cell ties all the steps together. For each patient:

```
1. Select best MRI and PET scans
2. Load NIfTI files (with RAS→LPS conversion)
3. N4 bias field correction on MRI
4. Skull stripping → brain MRI + brain mask
5. Register skull-stripped MRI → MNI template space
6. Transform brain mask → MNI space
7. Register PET → native MRI space
8. Transform PET → MNI space (using MRI→MNI transform)
9. Apply brain mask to both MRI and PET (in MNI space)
10. PET SUVR-like normalization
11. Optional PET smoothing
12. Resample to target shape (SimpleITK + center crop/pad)
13. Intensity normalization (percentile or z-score)
14. Save as .npy files
```

In [18]:
def process_single_patient(patient_id, input_base, output_base, mni_template, config):
    """
    Run the full preprocessing pipeline for one patient.
    
    Returns True if successful, False otherwise.
    """
    patient_input = os.path.join(input_base, patient_id)
    patient_output = os.path.join(output_base, patient_id)

    mri_out_path = os.path.join(patient_output, "mri_processed.npy")
    pet_out_path = os.path.join(patient_output, "pet_processed.npy")

    # Skip if already processed
    if os.path.exists(mri_out_path) and os.path.exists(pet_out_path):
        log(f"    Already processed, skipping.")
        return True

    # --- Step 1: Select best MRI and PET scans ---
    mri_path = choose_mri(os.path.join(patient_input, "MRI"))
    pet_path = choose_pet(os.path.join(patient_input, "PET"))

    if not mri_path:
        log(f"    No MRI found, skipping.")
        return False
    if not pet_path:
        log(f"    No PET found, skipping.")
        return False

    # --- Step 2: Load NIfTI files ---
    mri_sitk = nifti_to_sitk(mri_path)
    pet_sitk = nifti_to_sitk(pet_path)
    log(f"    MRI size: {mri_sitk.GetSize()}, PET size: {pet_sitk.GetSize()}")

    # --- Step 3: N4 bias field correction on MRI ---
    log(f"    Running N4 bias field correction...")
    mri_corrected = n4_bias_correction(mri_sitk)

    # --- Step 4: Skull stripping ---
    log(f"    Skull stripping ({config['skull_strip_method']})...")
    brain_mri, brain_mask = skull_strip(
        mri_corrected,
        method=config['skull_strip_method'],
        temp_dir=config['temp_dir']
    )

    # --- Step 5: Register skull-stripped MRI → MNI space ---
    log(f"    Registering MRI to MNI template...")
    mri_in_mni, mri_to_mni_transform = register_to_mni(brain_mri, mni_template)

    # --- Step 6: Transform brain mask to MNI space ---
    mask_in_mni = resample_with_transform(
        brain_mask, mni_template, mri_to_mni_transform, is_mask=True
    )

    # --- Step 7: Register PET → native MRI space ---
    log(f"    Registering PET to MRI...")
    pet_in_mri, _ = register_pet_to_mri(pet_sitk, mri_corrected)

    # --- Step 8: Transform PET → MNI space ---
    # PET is now aligned to native MRI, so we apply the same MRI→MNI transform
    pet_in_mni = resample_with_transform(
        pet_in_mri, mni_template, mri_to_mni_transform
    )

    # --- Step 9: Apply brain mask to both MRI and PET ---
    log(f"    Applying brain mask...")
    mri_masked = apply_brain_mask(mri_in_mni, mask_in_mni)
    pet_masked = apply_brain_mask(pet_in_mni, mask_in_mni)

    # --- Step 10: PET SUVR-like normalization ---
    # Convert to numpy for intensity operations
    mask_array = sitk_to_numpy(mask_in_mni)
    pet_array = sitk_to_numpy(pet_masked)
    log(f"    Applying PET SUVR normalization...")
    pet_array = normalize_pet_suvr(pet_array, mask_array)

    # --- Step 11: Optional PET smoothing ---
    if config['pet_smoothing_fwhm'] > 0:
        log(f"    Smoothing PET (FWHM={config['pet_smoothing_fwhm']}mm)...")
        pet_masked = smooth_pet(pet_masked, fwhm_mm=config['pet_smoothing_fwhm'])
        # Re-extract numpy after smoothing
        pet_array = sitk_to_numpy(pet_masked)
        pet_array = normalize_pet_suvr(pet_array, mask_array)

    # --- Step 12: Resample to target shape ---
    log(f"    Resampling to {config['target_shape']}...")
    mri_resampled = resample_to_target(
        mri_masked, config['target_shape'], config['target_spacing']
    )
    pet_resampled = resample_to_target(
        pet_masked, config['target_shape'], config['target_spacing']
    )

    # --- Step 13: Intensity normalization ---
    if config['normalization'] == 'percentile':
        mri_final = normalize_percentile(mri_resampled, 0.5, 99.5)
        pet_final = normalize_percentile(pet_resampled, 1.0, 99.0)
    else:
        mri_final = normalize_zscore(mri_resampled)
        pet_final = normalize_zscore(pet_resampled)

    # --- Step 14: Save ---
    os.makedirs(patient_output, exist_ok=True)
    np.save(mri_out_path, mri_final.astype(np.float32))
    np.save(pet_out_path, pet_final.astype(np.float32))

    log(f"    Saved: {config['target_shape']}")

    return True

## 15. Running the whole Pipeline

In [ ]:
config = {
    'target_shape': TARGET_SHAPE,
    'target_spacing': TARGET_SPACING,
    'skull_strip_method': SKULL_STRIP_METHOD,
    'normalization': NORMALIZATION_METHOD,
    'pet_smoothing_fwhm': PET_SMOOTHING_FWHM,
    'temp_dir': TEMP_DIR,
}

# GET PATIENT LIST
all_patients = sorted([
    d for d in os.listdir(INPUT_BASE)
    if os.path.isdir(os.path.join(INPUT_BASE, d))
])
patients_to_process = all_patients[:NUM_PATIENTS]

log(f"Total patients available: {len(all_patients)}")
log(f"Processing: {len(patients_to_process)} patients")
log(f"Target shape: {TARGET_SHAPE}")
log(f"Target spacing: {TARGET_SPACING}")
log(f"Skull stripping: {SKULL_STRIP_METHOD}")
log(f"Normalization: {NORMALIZATION_METHOD}")
log(f"PET smoothing: {PET_SMOOTHING_FWHM} mm FWHM")
log(f"Output: {OUTPUT_BASE}")
log(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log("")

# PHASE 1: EXTRACT ZIPS
log("PHASE 1: ZIP EXTRACTION")
extracted_count = extract_zips(patients_to_process, INPUT_BASE)
log("")

# PHASE 2: DICOM → NIfTI
log("PHASE 2: DICOM TO NIfTI CONVERSION")
nifti_count = convert_dicom_to_nifti(patients_to_process, INPUT_BASE)
log("")

# PHASE 3: PREPROCESSING (N4 + Skull Strip + MNI + PET Reg + Normalize)
log("PHASE 3: FULL PREPROCESSING")
log("")

processed = 0
skipped = 0
errors = 0

for idx, patient_id in enumerate(patients_to_process, 1):
    log(f"[{idx}/{len(patients_to_process)}] {patient_id}:")

    try:
        success = process_single_patient(
            patient_id, INPUT_BASE, OUTPUT_BASE, MNI_TEMPLATE, config
        )
        if success:
            processed += 1
        else:
            skipped += 1
    except Exception as e:
        log(f"    ERROR: {e}")
        errors += 1

    # Free memory after each patient
    gc.collect()

    if idx % 10 == 0:
        log(f"--- Progress: {processed} done, {skipped} skipped, {errors} errors ---")

# SUMMARY
log("")
log("=" * 90)
log("PIPELINE COMPLETE")
log(f"Patients processed:  {processed}")
log(f"Patients skipped:    {skipped}")
log(f"Errors:              {errors}")
log(f"ZIPs extracted:      {extracted_count}")
log(f"NIfTI files:         {nifti_count}")
log(f"Output shape:        {TARGET_SHAPE}")
log(f"Output directory:    {OUTPUT_BASE}")
log(f"Log file:            {LOG_FILE}")
log(f"End time:            {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

[2026-03-16 08:18:19] INFO: ==========================================================================================
[2026-03-16 08:18:19] INFO: ENHANCED MRI-PET PREPROCESSING PIPELINE
[2026-03-16 08:18:19] INFO: ==========================================================================================
[2026-03-16 08:18:19] INFO: Total patients available: 188
[2026-03-16 08:18:19] INFO: Processing: 188 patients
[2026-03-16 08:18:19] INFO: Target shape: (160, 192, 160)
[2026-03-16 08:18:19] INFO: Target spacing: (1.0, 1.0, 1.0)
[2026-03-16 08:18:19] INFO: Skull stripping: otsu
[2026-03-16 08:18:19] INFO: Normalization: percentile
[2026-03-16 08:18:19] INFO: PET smoothing: 4.0 mm FWHM
[2026-03-16 08:18:19] INFO: Output: D:\Organized_New_Only\Normal\Normal_Preprocessed_Enhanced
[2026-03-16 08:18:19] INFO: Start time: 2026-03-16 08:18:19
[2026-03-16 08:18:19] INFO: 
[2026-03-16 08:18:19] INFO: ==========================================================================================
[202

## 16. Quick Verification

In [19]:
# Quick check: load one processed patient and print stats
import matplotlib.pyplot as plt

# Pick the first successfully processed patient
for pid in patients_to_process:
    mri_file = os.path.join(OUTPUT_BASE, pid, "mri_processed.npy")
    pet_file = os.path.join(OUTPUT_BASE, pid, "pet_processed.npy")
    if os.path.exists(mri_file) and os.path.exists(pet_file):
        mri = np.load(mri_file)
        pet = np.load(pet_file)

        print(f"Patient: {pid}")
        print(f"MRI shape: {mri.shape}, min: {mri.min():.3f}, max: {mri.max():.3f}, mean: {mri.mean():.3f}")
        print(f"PET shape: {pet.shape}, min: {pet.min():.3f}, max: {pet.max():.3f}, mean: {pet.mean():.3f}")

        # Show middle axial slice
        mid = mri.shape[2] // 2
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(mri[:, :, mid].T, cmap='gray', origin='lower')
        axes[0].set_title(f"MRI (axial slice {mid})")
        axes[0].axis('off')
        axes[1].imshow(pet[:, :, mid].T, cmap='hot', origin='lower')
        axes[1].set_title(f"PET (axial slice {mid})")
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()
        break
else:
    print("No processed patients found.")

NameError: name 'patients_to_process' is not defined

## 17. Generate QC Images (PNG)

For each processed patient, generate a simple **3-row x 3-column** PNG showing:
- **Row 1:** MRI (Axial, Coronal, Sagittal)
- **Row 2:** PET (Axial, Coronal, Sagittal)
- **Row 3:** Overlay (PET on top of MRI)

In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def rescale_for_display(volume):
    """Rescale volume to [0, 1] using 1st-99th percentile clipping."""
    v = volume.copy().astype(np.float32)
    p1 = np.percentile(v, 1)
    p99 = np.percentile(v, 99)
    v = np.clip(v, p1, p99)
    v = (v - p1) / (p99 - p1 + 1e-8)
    return v


def get_centre_slices(volume):
    """Return centre axial, coronal, sagittal slices."""
    x, y, z = volume.shape[0] // 2, volume.shape[1] // 2, volume.shape[2] // 2
    axial    = volume[:, :, z].T   # x-y plane
    coronal  = volume[:, y, :].T   # x-z plane
    sagittal = volume[x, :, :].T   # y-z plane
    return axial, coronal, sagittal


def save_qc_png(mri_data, pet_data, patient_id, output_dir):
    """
    Save a simple 3x3 QC figure as PNG.
    Row 0: MRI  (gray)   — Axial, Coronal, Sagittal
    Row 1: PET  (hot)    — Axial, Coronal, Sagittal
    Row 2: Overlay       — PET on MRI with transparency
    """
    mri = rescale_for_display(mri_data)
    pet = rescale_for_display(pet_data)

    mri_slices = get_centre_slices(mri)
    pet_slices = get_centre_slices(pet)
    view_names = ["Axial", "Coronal", "Sagittal"]

    fig, axes = plt.subplots(3, 3, figsize=(12, 12))

    for col in range(3):
        # Row 0 — MRI
        axes[0, col].imshow(mri_slices[col], cmap="gray", origin="lower")
        axes[0, col].set_title(f"MRI {view_names[col]}")
        axes[0, col].axis("off")

        # Row 1 — PET
        axes[1, col].imshow(pet_slices[col], cmap="hot", origin="lower")
        axes[1, col].set_title(f"PET {view_names[col]}")
        axes[1, col].axis("off")

        # Row 2 — Overlay (PET over MRI)
        axes[2, col].imshow(mri_slices[col], cmap="gray", origin="lower")
        axes[2, col].imshow(pet_slices[col], cmap="hot", alpha=0.4, origin="lower")
        axes[2, col].set_title(f"Overlay {view_names[col]}")
        axes[2, col].axis("off")

    fig.suptitle(
        f"Patient: {patient_id}  |  Shape: {mri_data.shape}",
        fontsize=14
    )
    fig.tight_layout()

    out_path = os.path.join(output_dir, f"{patient_id}_qc.png")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path


# --- Generate QC PNGs for all processed patients ---
print("Generating QC images...")
qc_generated = 0

for patient_id in patients_to_process:
    patient_dir = os.path.join(OUTPUT_BASE, patient_id)
    mri_file = os.path.join(patient_dir, "mri_processed.npy")
    pet_file = os.path.join(patient_dir, "pet_processed.npy")

    # Skip if not processed or QC already exists
    if not os.path.exists(mri_file) or not os.path.exists(pet_file):
        continue
    qc_path = os.path.join(patient_dir, f"{patient_id}_qc.png")
    if os.path.exists(qc_path):
        qc_generated += 1
        continue

    mri_data = np.load(mri_file)
    pet_data = np.load(pet_file)
    save_qc_png(mri_data, pet_data, patient_id, patient_dir)
    qc_generated += 1

    del mri_data, pet_data

print(f"Done — {qc_generated} QC images saved to {OUTPUT_BASE}")

Generating QC images...


NameError: name 'patients_to_process' is not defined